<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/station1_integrity_gate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth -q

## case1

In [1]:
!nvidia-smi

Mon Oct  5 11:51:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#!/usr/bin/env python3
# ============================================================================
# CERTIFIED INFERENCE & DRIFT VERIFICATION SCRIPT (SILENT UNSLOTH & CLEAN OUTPUT)
# MODEL REPO : frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified
# BACKBONE   : frankmorales2020/gemma-4-e4b-unesco-optimized
# DATASET    : CIFAR-100 (REAL PIXELS EVALUATION ON UNIFIED TEST SET)
# ============================================================================

import builtins
import contextlib
import hashlib
import logging
import os
import random
import sys
import warnings
from typing import List

# 0. Global Silencer & Warning Supression
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["UNSLOTH_SILENT"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


@contextlib.contextmanager
def silence_fd():
    orig_print = builtins.print
    builtins.print = lambda *args, **kwargs: None
    sys.stdout.flush()
    sys.stderr.flush()
    try:
        orig_stdout_fd = os.dup(1)
        orig_stderr_fd = os.dup(2)
        has_fd = True
    except Exception:
        has_fd = False

    if has_fd:
        devnull_fd = os.open(os.devnull, os.O_WRONLY)
        os.dup2(devnull_fd, 1)
        os.dup2(devnull_fd, 2)
    try:
        yield
    finally:
        builtins.print = orig_print
        if has_fd:
            os.dup2(orig_stdout_fd, 1)
            os.dup2(orig_stderr_fd, 2)
            os.close(orig_stdout_fd)
            os.close(orig_stderr_fd)
            os.close(devnull_fd)


from google.colab import userdata
from huggingface_hub import hf_hub_download
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torchvision

# 1. Setup & Environment
HF_TOKEN = userdata.get("HF_TOKEN")
REPO_ID = "frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified"
BACKBONE_ID = "frankmorales2020/gemma-4-e4b-unesco-optimized"
CHECKPOINT_FILENAME = "topo_cbp_gemma4_cifar100_best_certified.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 4

print("=" * 80, flush=True)
print(f"📥 DOWNLOADING CERTIFIED CHECKPOINT FROM HF: {REPO_ID}", flush=True)
print(f"   Target Device: {DEVICE}", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    checkpoint_path = hf_hub_download(
        repo_id=REPO_ID, filename=CHECKPOINT_FILENAME, token=HF_TOKEN
    )
print(f"✅ Checkpoint cached at: {checkpoint_path}", flush=True)

# 2. Load Gemma-4 E4B Backbone in Deep Silence
print("\n" + "=" * 80, flush=True)
print(f"📥 LOADING GEMMA-4 E4B MULTIMODAL BACKBONE: {BACKBONE_ID}", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    from unsloth import FastVisionModel

    base_model, processor = FastVisionModel.from_pretrained(
        BACKBONE_ID,
        load_in_4bit=True,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    FastVisionModel.for_inference(base_model)

lm_backbone = (
    base_model.language_model
    if hasattr(base_model, "language_model")
    else base_model
)
embed_layer = lm_backbone.get_input_embeddings()
hidden_size = 2560


# 3. Multimodal Classifier Definition
class GemmaE4BVisionClassifier(nn.Module):
    def __init__(self, base_model, hidden_size):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.current_task = "A"

    def forward(self, **inputs):
        inputs["output_hidden_states"] = True
        outputs = self.base_model(**inputs)

        if hasattr(outputs, "hidden_states") and outputs.hidden_states is not None:
            hidden_states = outputs.hidden_states[-1]
        elif hasattr(outputs, "last_hidden_state"):
            hidden_states = outputs.last_hidden_state
        else:
            raise AttributeError(
                "Unable to retrieve hidden states from multimodal model."
            )

        hidden_states = hidden_states.float()
        attention_mask = inputs.get("attention_mask", None)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(
                min=1e-8
            )
        else:
            pooled = hidden_states.mean(dim=1)

        head = getattr(self, f"classifier_{self.current_task}")
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ("A", "B", "C")
        self.current_task = task


# 4. Instantiate Model & Restore Certified Weights
model = GemmaE4BVisionClassifier(base_model, hidden_size).to(DEVICE)
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

model.classifier_A.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_A"].items()}
)
model.classifier_B.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_B"].items()}
)
model.classifier_C.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_C"].items()}
)

with torch.no_grad():
    embed_layer.weight.copy_(
        state["embed_weight"].to(DEVICE, dtype=embed_layer.weight.dtype)
    )

model.eval()
print(
    "✅ Certified classification heads and embedding representations restored.",
    flush=True,
)

# 5. Verify Prime Subspace Bit-Level Invariance
print("\n" + "=" * 80, flush=True)
print("🔒 AUDITING PRIME TOPOLOGICAL ANCHORS & BIT-LEVEL INVARIANCE", flush=True)
print("=" * 80, flush=True)

anchor_indices = state["prime_anchors"]
saved_hash = state["state_hash"]

stacked_anchors = torch.stack(
    [embed_layer.weight[idx].detach().float() for idx in anchor_indices]
)
current_hash = hashlib.sha256(
    stacked_anchors.cpu().numpy().tobytes()
).hexdigest()[:16]

is_invariant = current_hash == saved_hash
print(f"  Clamped Prime Coordinates : {anchor_indices}", flush=True)
print(f"  Safety Constant (Λ)       : {state['safety_constant']:.10f}", flush=True)
print(f"  Saved SHA-256 Fingerprint : {saved_hash}", flush=True)
print(f"  Live SHA-256 Fingerprint  : {current_hash}", flush=True)
print(
    "  Verification Status       :"
    f" {'✅ BIT-LEVEL INVARIANCE VERIFIED' if is_invariant else '❌ DRIFT DETECTED'}",
    flush=True,
)

# 6. Prepare CIFAR-100 Real-Pixel Test Set
print("\n" + "=" * 80, flush=True)
print("📚 PREPARING CIFAR-100 UNIFIED TEST SET (300 RAW IMAGES)", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    raw_testset = torchvision.datasets.CIFAR100(
        root="./data_cifar100", train=False, download=True
    )

SHARED_ANIMALS = [4, 30, 55, 72, 95]
SHARED_VEHICLES = [8, 13, 48, 58, 90]
TEST_SAMPLES = 300


def extract_raw_image_split(
    dataset,
    class_0: List[int],
    class_1: List[int],
    num_samples: int,
    seed_val: int,
):
    random.seed(seed_val)
    images_pil, labels = [], []
    per_class_0 = (num_samples // 2) // len(class_0)
    per_class_1 = (num_samples // 2) // len(class_1)
    targets = np.array(dataset.targets)

    for cls in class_0:
        idx = np.where(targets == cls)[0].tolist()
        for i in random.sample(idx, min(per_class_0, len(idx))):
            img_raw, _ = dataset[i]
            images_pil.append(
                img_raw.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BICUBIC)
            )
            labels.append(0)

    for cls in class_1:
        idx = np.where(targets == cls)[0].tolist()
        for i in random.sample(idx, min(per_class_1, len(idx))):
            img_raw, _ = dataset[i]
            images_pil.append(
                img_raw.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BICUBIC)
            )
            labels.append(1)

    combined = list(zip(images_pil, labels))
    random.shuffle(combined)
    images_pil, labels = zip(*combined)
    return list(images_pil), list(labels)


test_imgs, test_labels = extract_raw_image_split(
    raw_testset, SHARED_ANIMALS, SHARED_VEHICLES, TEST_SAMPLES, seed_val=999
)


class MultimodalImageDataset(torch.utils.data.Dataset):
    def __init__(self, images: List[Image.Image], labels: List[int]):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


def multimodal_collate_fn(batch):
    images, labels = zip(*batch)
    messages = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Identify this entity."},
                ],
            }
        ]
        for img in images
    ]
    texts = [
        processor.apply_chat_template(m, add_generation_prompt=True)
        for m in messages
    ]
    batched_images = [[img] for img in images]

    batch_inputs = processor(
        text=texts, images=batched_images, return_tensors="pt", padding=True
    )
    if (
        "pixel_values" in batch_inputs
        and batch_inputs["pixel_values"].dtype != torch.bfloat16
    ):
        batch_inputs["pixel_values"] = batch_inputs["pixel_values"].to(
            dtype=torch.bfloat16
        )
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    return batch_inputs, labels_tensor


test_loader = torch.utils.data.DataLoader(
    MultimodalImageDataset(test_imgs, test_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=multimodal_collate_fn,
)


# 7. Execute Inference Evaluation Across Tasks
@torch.no_grad()
def evaluate_task(task_label: str) -> float:
    model.switch_task(task_label)
    model.eval()
    all_preds, all_labels = [], []
    for batch_inputs, labels in test_loader:
        batch_inputs = {
            k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
            for k, v in batch_inputs.items()
        }
        labels = labels.to(DEVICE)
        logits = model(**batch_inputs)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


print("\n" + "=" * 80, flush=True)
print("🚀 RUNNING SEQUENTIAL TASK INFERENCE (REAL PIXELS)", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    acc_A = evaluate_task("A")
print(f"  [EVAL] Task A (Animal vs Vehicle)    : {acc_A * 100:.2f}%", flush=True)

with silence_fd():
    acc_B = evaluate_task("B")
print(f"  [EVAL] Task B (Natural vs Man-Made)  : {acc_B * 100:.2f}%", flush=True)

with silence_fd():
    acc_C = evaluate_task("C")
print(f"  [EVAL] Task C (Living vs Non-Living) : {acc_C * 100:.2f}%", flush=True)

print("\n" + "=" * 80, flush=True)
print("📊 FINAL BENCHMARK SUMMARY", flush=True)
print("=" * 80, flush=True)
print(f"  Task A Accuracy : {acc_A * 100:.2f}%", flush=True)
print(f"  Task B Accuracy : {acc_B * 100:.2f}%", flush=True)
print(f"  Task C Accuracy : {acc_C * 100:.2f}%", flush=True)
print(f"  Mean Accuracy   : {np.mean([acc_A, acc_B, acc_C]) * 100:.2f}%", flush=True)
print(
    f"  Bit Invariance  : {'PASSED (Zero Drift)' if is_invariant else 'FAILED'}",
    flush=True,
)
print("=" * 80, flush=True)

📥 DOWNLOADING CERTIFIED CHECKPOINT FROM HF: frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified
   Target Device: cuda
✅ Checkpoint cached at: /root/.cache/huggingface/hub/models--frankmorales2020--topo-cbp-gemma-4-e4b-cifar100-certified/snapshots/c8131afb36120006f954c52e0c4648b91ee2b291/topo_cbp_gemma4_cifar100_best_certified.pt

📥 LOADING GEMMA-4 E4B MULTIMODAL BACKBONE: frankmorales2020/gemma-4-e4b-unesco-optimized


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

✅ Certified classification heads and embedding representations restored.

🔒 AUDITING PRIME TOPOLOGICAL ANCHORS & BIT-LEVEL INVARIANCE
  Clamped Prime Coordinates : [2, 3, 5, 7, 11, 13]
  Safety Constant (Λ)       : 0.9785142874
  Saved SHA-256 Fingerprint : ab9f23d2bc947712
  Live SHA-256 Fingerprint  : ab9f23d2bc947712
  Verification Status       : ✅ BIT-LEVEL INVARIANCE VERIFIED

📚 PREPARING CIFAR-100 UNIFIED TEST SET (300 RAW IMAGES)


100%|██████████| 169M/169M [17:26<00:00, 162kB/s]



🚀 RUNNING SEQUENTIAL TASK INFERENCE (REAL PIXELS)
  [EVAL] Task A (Animal vs Vehicle)    : 99.67%
  [EVAL] Task B (Natural vs Man-Made)  : 99.67%
  [EVAL] Task C (Living vs Non-Living) : 100.00%

📊 FINAL BENCHMARK SUMMARY
  Task A Accuracy : 99.67%
  Task B Accuracy : 99.67%
  Task C Accuracy : 100.00%
  Mean Accuracy   : 99.78%
  Bit Invariance  : PASSED (Zero Drift)


In [3]:
!nvidia-smi

Mon Oct  5 12:19:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   65C    P0             30W /   72W |   13970MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## case2

In [1]:
!nvidia-smi

Mon Oct  5 12:21:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   58C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#!/usr/bin/env python3
# ============================================================================
# STATION 1: WEIGHTS & INVARIANT SUBSPACE PLANE (SOMALA)
# CERTIFIED INFERENCE & DRIFT VERIFICATION GATE FOR IAST CONTRACT
# REPO : frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified
# ============================================================================

import builtins
import contextlib
import hashlib
import json
import logging
import os
import random
import sys
import warnings
from typing import List, Dict, Any

# 0. Global Silencer & Warning Supression
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["UNSLOTH_SILENT"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


@contextlib.contextmanager
def silence_fd():
    orig_print = builtins.print
    builtins.print = lambda *args, **kwargs: None
    sys.stdout.flush()
    sys.stderr.flush()
    try:
        orig_stdout_fd = os.dup(1)
        orig_stderr_fd = os.dup(2)
        has_fd = True
    except Exception:
        has_fd = False

    if has_fd:
        devnull_fd = os.open(os.devnull, os.O_WRONLY)
        os.dup2(devnull_fd, 1)
        os.dup2(devnull_fd, 2)
    try:
        yield
    finally:
        builtins.print = orig_print
        if has_fd:
            os.dup2(orig_stdout_fd, 1)
            os.dup2(orig_stderr_fd, 2)
            os.close(orig_stdout_fd)
            os.close(orig_stderr_fd)
            os.close(devnull_fd)


from google.colab import userdata
from huggingface_hub import hf_hub_download
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torchvision

# 1. Setup & Environment
HF_TOKEN = userdata.get("HF_TOKEN")
REPO_ID = "frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified"
BACKBONE_ID = "frankmorales2020/gemma-4-e4b-unesco-optimized"
CHECKPOINT_FILENAME = "topo_cbp_gemma4_cifar100_best_certified.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 4

print("=" * 80, flush=True)
print("🛡️ STATION 1 (SOMALA): WEIGHTS & INVARIANT SUBSPACE GATE", flush=True)
print(f"📥 DOWNLOADING CERTIFIED CHECKPOINT FROM HF: {REPO_ID}", flush=True)
print(f"   Target Device: {DEVICE}", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    checkpoint_path = hf_hub_download(
        repo_id=REPO_ID, filename=CHECKPOINT_FILENAME, token=HF_TOKEN
    )
print(f"✅ Checkpoint cached at: {checkpoint_path}", flush=True)

# 2. Load Gemma-4 E4B Backbone in Deep Silence
print("\n" + "=" * 80, flush=True)
print(f"📥 LOADING GEMMA-4 E4B MULTIMODAL BACKBONE: {BACKBONE_ID}", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    from unsloth import FastVisionModel

    base_model, processor = FastVisionModel.from_pretrained(
        BACKBONE_ID,
        load_in_4bit=True,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    FastVisionModel.for_inference(base_model)

lm_backbone = (
    base_model.language_model
    if hasattr(base_model, "language_model")
    else base_model
)
embed_layer = lm_backbone.get_input_embeddings()
hidden_size = 2560


# 3. Multimodal Classifier Definition (Station 1 Execution Plane)
class Station1VisionClassifier(nn.Module):
    def __init__(self, base_model, hidden_size):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.float32)
        self.current_task = "A"

    def forward(self, **inputs):
        inputs["output_hidden_states"] = True
        outputs = self.base_model(**inputs)

        if hasattr(outputs, "hidden_states") and outputs.hidden_states is not None:
            hidden_states = outputs.hidden_states[-1]
        elif hasattr(outputs, "last_hidden_state"):
            hidden_states = outputs.last_hidden_state
        else:
            raise AttributeError(
                "Unable to retrieve hidden states from multimodal model."
            )

        hidden_states = hidden_states.float()
        attention_mask = inputs.get("attention_mask", None)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(
                min=1e-8
            )
        else:
            pooled = hidden_states.mean(dim=1)

        head = getattr(self, f"classifier_{self.current_task}")
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ("A", "B", "C")
        self.current_task = task


# 4. Instantiate Model & Restore Certified Weights
model = Station1VisionClassifier(base_model, hidden_size).to(DEVICE)
state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

model.classifier_A.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_A"].items()}
)
model.classifier_B.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_B"].items()}
)
model.classifier_C.load_state_dict(
    {k: v.to(DEVICE) for k, v in state["classifier_C"].items()}
)

with torch.no_grad():
    embed_layer.weight.copy_(
        state["embed_weight"].to(DEVICE, dtype=embed_layer.weight.dtype)
    )

model.eval()
print(
    "✅ Station 1 classification heads and prime-anchored embeddings restored.",
    flush=True,
)

# 5. Verify Prime Subspace Bit-Level Invariance (Activation Drift Plane Audit)
print("\n" + "=" * 80, flush=True)
print("🔒 STATION 1 AUDIT: PRIME TOPOLOGICAL ANCHORS & SHA-256 INTEGRITY", flush=True)
print("=" * 80, flush=True)

anchor_indices = state["prime_anchors"]
saved_hash = state["state_hash"]

stacked_anchors = torch.stack(
    [embed_layer.weight[idx].detach().float() for idx in anchor_indices]
)
current_hash = hashlib.sha256(
    stacked_anchors.cpu().numpy().tobytes()
).hexdigest()[:16]

is_invariant = current_hash == saved_hash
delta_drift = 0.0 if is_invariant else 1.0

print(f"  Clamped Prime Coordinates : {anchor_indices}", flush=True)
print(f"  Safety Constant (Λ)       : {state['safety_constant']:.10f}", flush=True)
print(f"  Saved SHA-256 Fingerprint : {saved_hash}", flush=True)
print(f"  Live SHA-256 Fingerprint  : {current_hash}", flush=True)
print(f"  Max Coordinate Drift      : Δ_drift = {delta_drift:.10f}", flush=True)
print(
    "  Station 1 Gate Status     :"
    f" {'✅ AD_PASS (WEIGHTS INVARIANT)' if is_invariant else '❌ AD_FAIL (DRIFT DETECTED)'}",
    flush=True,
)

if not is_invariant:
    raise RuntimeError("CRITICAL: Station 1 weight invariant subspace integrity violated!")

# 6. Prepare CIFAR-100 Real-Pixel Test Set
print("\n" + "=" * 80, flush=True)
print("📚 PREPARING CIFAR-100 UNIFIED TEST SET (300 RAW IMAGES)", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    raw_testset = torchvision.datasets.CIFAR100(
        root="./data_cifar100", train=False, download=True
    )

SHARED_ANIMALS = [4, 30, 55, 72, 95]
SHARED_VEHICLES = [8, 13, 48, 58, 90]
TEST_SAMPLES = 300


def extract_raw_image_split(
    dataset,
    class_0: List[int],
    class_1: List[int],
    num_samples: int,
    seed_val: int,
):
    random.seed(seed_val)
    images_pil, labels = [], []
    per_class_0 = (num_samples // 2) // len(class_0)
    per_class_1 = (num_samples // 2) // len(class_1)
    targets = np.array(dataset.targets)

    for cls in class_0:
        idx = np.where(targets == cls)[0].tolist()
        for i in random.sample(idx, min(per_class_0, len(idx))):
            img_raw, _ = dataset[i]
            images_pil.append(
                img_raw.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BICUBIC)
            )
            labels.append(0)

    for cls in class_1:
        idx = np.where(targets == cls)[0].tolist()
        for i in random.sample(idx, min(per_class_1, len(idx))):
            img_raw, _ = dataset[i]
            images_pil.append(
                img_raw.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BICUBIC)
            )
            labels.append(1)

    combined = list(zip(images_pil, labels))
    random.shuffle(combined)
    images_pil, labels = zip(*combined)
    return list(images_pil), list(labels)


test_imgs, test_labels = extract_raw_image_split(
    raw_testset, SHARED_ANIMALS, SHARED_VEHICLES, TEST_SAMPLES, seed_val=999
)


class MultimodalImageDataset(torch.utils.data.Dataset):
    def __init__(self, images: List[Image.Image], labels: List[int]):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


def multimodal_collate_fn(batch):
    images, labels = zip(*batch)
    messages = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Identify this entity."},
                ],
            }
        ]
        for img in images
    ]
    texts = [
        processor.apply_chat_template(m, add_generation_prompt=True)
        for m in messages
    ]
    batched_images = [[img] for img in images]

    batch_inputs = processor(
        text=texts, images=batched_images, return_tensors="pt", padding=True
    )
    if (
        "pixel_values" in batch_inputs
        and batch_inputs["pixel_values"].dtype != torch.bfloat16
    ):
        batch_inputs["pixel_values"] = batch_inputs["pixel_values"].to(
            dtype=torch.bfloat16
        )
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    return batch_inputs, labels_tensor


test_loader = torch.utils.data.DataLoader(
    MultimodalImageDataset(test_imgs, test_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=multimodal_collate_fn,
)


# 7. Execute Sequential Task Evaluation
@torch.no_grad()
def evaluate_task(task_label: str) -> float:
    model.switch_task(task_label)
    model.eval()
    all_preds, all_labels = [], []
    for batch_inputs, labels in test_loader:
        batch_inputs = {
            k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
            for k, v in batch_inputs.items()
        }
        labels = labels.to(DEVICE)
        logits = model(**batch_inputs)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


print("\n" + "=" * 80, flush=True)
print("🚀 STATION 1: EXECUTING SEQUENTIAL REAL-PIXEL EVALUATION", flush=True)
print("=" * 80, flush=True)

with silence_fd():
    acc_A = evaluate_task("A")
print(f"  [EVAL] Task A (Animal vs Vehicle)    : {acc_A * 100:.2f}%", flush=True)

with silence_fd():
    acc_B = evaluate_task("B")
print(f"  [EVAL] Task B (Natural vs Man-Made)  : {acc_B * 100:.2f}%", flush=True)

with silence_fd():
    acc_C = evaluate_task("C")
print(f"  [EVAL] Task C (Living vs Non-Living) : {acc_C * 100:.2f}%", flush=True)

mean_acc = float(np.mean([acc_A, acc_B, acc_C]))

# 8. Generate Station 1 Handoff Telemetry Payload
station_1_payload: Dict[str, Any] = {
    "station": "STATION_1_SOMALA",
    "status": "AD_PASS",
    "model_repo": REPO_ID,
    "state_hash": current_hash,
    "delta_drift": delta_drift,
    "metrics": {
        "task_a_acc": float(acc_A),
        "task_b_acc": float(acc_B),
        "task_c_acc": float(acc_C),
        "mean_acc": mean_acc,
    },
    "contract_compliance": "IAST-CANDIDATE-HANDOFF-CONTRACT-001",
}

print("\n" + "=" * 80, flush=True)
print("📊 STATION 1 COMPLIANCE & HANDOFF TELEMETRY", flush=True)
print("=" * 80, flush=True)
print(json.dumps(station_1_payload, indent=2))
print("=" * 80, flush=True)
print("✅ Station 1 execution complete. Ready for Station 2 / H-Boundary sync.", flush=True)

🛡️ STATION 1 (SOMALA): WEIGHTS & INVARIANT SUBSPACE GATE
📥 DOWNLOADING CERTIFIED CHECKPOINT FROM HF: frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified
   Target Device: cuda
✅ Checkpoint cached at: /root/.cache/huggingface/hub/models--frankmorales2020--topo-cbp-gemma-4-e4b-cifar100-certified/snapshots/c8131afb36120006f954c52e0c4648b91ee2b291/topo_cbp_gemma4_cifar100_best_certified.pt

📥 LOADING GEMMA-4 E4B MULTIMODAL BACKBONE: frankmorales2020/gemma-4-e4b-unesco-optimized


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

✅ Station 1 classification heads and prime-anchored embeddings restored.

🔒 STATION 1 AUDIT: PRIME TOPOLOGICAL ANCHORS & SHA-256 INTEGRITY
  Clamped Prime Coordinates : [2, 3, 5, 7, 11, 13]
  Safety Constant (Λ)       : 0.9785142874
  Saved SHA-256 Fingerprint : ab9f23d2bc947712
  Live SHA-256 Fingerprint  : ab9f23d2bc947712
  Max Coordinate Drift      : Δ_drift = 0.0000000000
  Station 1 Gate Status     : ✅ AD_PASS (WEIGHTS INVARIANT)

📚 PREPARING CIFAR-100 UNIFIED TEST SET (300 RAW IMAGES)

🚀 STATION 1: EXECUTING SEQUENTIAL REAL-PIXEL EVALUATION
  [EVAL] Task A (Animal vs Vehicle)    : 99.67%
  [EVAL] Task B (Natural vs Man-Made)  : 99.67%
  [EVAL] Task C (Living vs Non-Living) : 100.00%

📊 STATION 1 COMPLIANCE & HANDOFF TELEMETRY
{
  "station": "STATION_1_SOMALA",
  "status": "AD_PASS",
  "model_repo": "frankmorales2020/topo-cbp-gemma-4-e4b-cifar100-certified",
  "state_hash": "ab9f23d2bc947712",
  "delta_drift": 0.0,
  "metrics": {
    "task_a_acc": 0.9966666666666667,
    "task_b_